In [1]:
from pathlib import Path
import json

import numpy as np
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

from data_utils import load_data_for_n, get_fold_split, N_LEVELS

MODELS_DIR = Path("../Models/MIL_Linear")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Setup done. N-levels:", N_LEVELS)

Setup done. N-levels: [1, 2, 4, 8, 16, 32, 64]


In [2]:
def pool_bag(bag_vectors, method="max"):
    """
    Collapse one speaker's chunks into a single vector.

    bag_vectors : ndarray of shape (n_chunks, 768)
    method      : "max"  -> MIL / selection  ("did ANY chunk light up here?")
                  "mean" -> aggregation      ("what's the typical level here?")

    Returns: ndarray of shape (768,)
    """
    if method == "max":
        return bag_vectors.max(axis=0)
    elif method == "mean":
        return bag_vectors.mean(axis=0)
    else:
        raise ValueError(f"Unknown pooling method: {method}")


# --- quick check on fake data so we can see it behaving ---
fake = np.array([
    [0.2, 0.1, 0.9, 0.3],
    [0.3, 0.2, 0.1, 0.2],
    [0.1, 0.4, 0.2, 0.1],
])
print("fake bag (3 chunks x 4 dims):\n", fake)
print("max-pooled :", pool_bag(fake, "max"))
print("mean-pooled:", pool_bag(fake, "mean"))

fake bag (3 chunks x 4 dims):
 [[0.2 0.1 0.9 0.3]
 [0.3 0.2 0.1 0.2]
 [0.1 0.4 0.2 0.1]]
max-pooled : [0.3 0.4 0.9 0.3]
mean-pooled: [0.2        0.23333333 0.4        0.2       ]


In [3]:
def build_bags(X, y, speaker, method="max"):
    """
    Turn flat per-chunk arrays into ONE pooled vector per speaker.

    X       : (n_chunks, 768)  — all chunks
    y       : (n_chunks,)      — label repeated for each chunk
    speaker : (n_chunks,)      — which speaker each chunk came from

    Returns:
        X_bags : (n_speakers, 768)  — one pooled vector per speaker
        y_bags : (n_speakers,)      — one label per speaker
    """
    X_bags, y_bags = [], []

    for spk in np.unique(speaker):
        mask = (speaker == spk)
        X_bags.append(pool_bag(X[mask], method=method))
        y_bags.append(y[mask][0])   # all chunks of a speaker share one label

    return np.array(X_bags), np.array(y_bags)


# --- sanity check on real data at N=8 ---
X, y, fold, speaker = load_data_for_n(8)
print("Raw chunks :", X.shape, "| unique speakers:", len(np.unique(speaker)))

X_bags, y_bags = build_bags(X, y, speaker, method="max")
print("Pooled bags:", X_bags.shape, "| labels:", y_bags.shape)
print("Label counts — HC(0):", (y_bags == 0).sum(), " PT(1):", (y_bags == 1).sum())

Raw chunks : (928, 768) | unique speakers: 116
Pooled bags: (116, 768) | labels: (116,)
Label counts — HC(0): 52  PT(1): 64


In [4]:
def run_pooled_cv(n, method="max", C=1.0):
    """
    5-fold person-independent CV for pooled-bag Logistic Regression.

    Pool each speaker's chunks into one vector, then classify speakers directly.
    No majority vote needed — there is already exactly one prediction per person.

    Returns per-fold F1s (needed later for significance testing) plus summary stats.
    """
    X, y, fold, speaker = load_data_for_n(n)

    fold_accs, fold_f1s = [], []

    for fold_number in [1, 2, 3, 4, 5]:
        # split at the CHUNK level first (keeps person-independence)
        train_mask = (fold != fold_number)
        test_mask  = (fold == fold_number)

        X_tr, y_tr = build_bags(X[train_mask], y[train_mask], speaker[train_mask], method)
        X_te, y_te = build_bags(X[test_mask],  y[test_mask],  speaker[test_mask],  method)

        clf = LogisticRegression(C=C, max_iter=1000)
        clf.fit(X_tr, y_tr)
        preds = clf.predict(X_te).astype(int)

        fold_accs.append((preds == y_te).mean())
        fold_f1s.append(f1_score(y_te, preds))

    return {
        "acc_mean": np.mean(fold_accs), "acc_std": np.std(fold_accs),
        "f1_mean":  np.mean(fold_f1s),  "f1_std":  np.std(fold_f1s),
        "fold_f1s": fold_f1s,      # keep per-fold values for the t-test later
        "fold_accs": fold_accs,
    }


# --- test at N=8, both pooling methods ---
for m in ["max", "mean"]:
    r = run_pooled_cv(n=8, method=m)
    print(f"N=8 {m:>4}-pool + LR: Acc={r['acc_mean']:.4f}±{r['acc_std']:.4f}, "
          f"F1={r['f1_mean']:.4f}±{r['f1_std']:.4f}")

N=8  max-pool + LR: Acc=0.8971±0.0573, F1=0.9098±0.0426
N=8 mean-pool + LR: Acc=0.9058±0.0622, F1=0.9172±0.0472


In [5]:
results_pooled = {"max": {}, "mean": {}}

for method in ["max", "mean"]:
    print(f"\n===== {method.upper()}-POOLING + LR =====")
    for n in tqdm(N_LEVELS, desc=f"{method}-pool", leave=False):
        results_pooled[method][n] = run_pooled_cv(n=n, method=method)

# Comparison table
print(f"\n{'N':>4} | {'max-pool F1':>16} | {'mean-pool F1':>16}")
print("-" * 44)
for n in N_LEVELS:
    rmax = results_pooled["max"][n]
    rmean = results_pooled["mean"][n]
    print(f"{n:>4} | {rmax['f1_mean']*100:6.2f}% ± {rmax['f1_std']*100:4.2f}% "
          f"| {rmean['f1_mean']*100:6.2f}% ± {rmean['f1_std']*100:4.2f}%")

# Save (convert numpy types so json is happy)
def clean(d):
    return {k: (float(v) if isinstance(v, (np.floating, float)) else
                [float(x) for x in v] if isinstance(v, list) else v)
            for k, v in d.items()}

with open(MODELS_DIR / "results_summary.json", "w") as f:
    json.dump({m: {str(n): clean(r) for n, r in res.items()}
               for m, res in results_pooled.items()}, f, indent=2)

print("\nSaved to Models/MIL_Linear/results_summary.json")


===== MAX-POOLING + LR =====



===== MEAN-POOLING + LR =====



   N |      max-pool F1 |     mean-pool F1
--------------------------------------------
   1 |  91.93% ± 5.60% |  91.93% ± 5.60%
   2 |  92.39% ± 2.75% |  91.71% ± 3.38%
   4 |  92.80% ± 5.71% |  91.07% ± 6.10%
   8 |  90.98% ± 4.26% |  91.72% ± 4.72%
  16 |  91.08% ± 6.29% |  91.07% ± 6.10%
  32 |  88.88% ± 6.77% |  89.48% ± 8.54%
  64 |  88.05% ± 7.36% |  90.30% ± 7.28%

Saved to Models/MIL_Linear/results_summary.json


In [6]:
from scipy import stats

print(f"{'N':>4} | {'max F1':>8} | {'mean F1':>8} | {'diff':>7} | {'p (two-sided)':>14}")
print("-" * 58)

for n in N_LEVELS:
    max_f1s  = np.array(results_pooled["max"][n]["fold_f1s"])
    mean_f1s = np.array(results_pooled["mean"][n]["fold_f1s"])
    diff = max_f1s.mean() - mean_f1s.mean()

    # paired: same folds, same data, only the pooling operator differs
    t_stat, p_val = stats.ttest_rel(max_f1s, mean_f1s)

    flag = "*" if p_val < 0.05 else ""
    print(f"{n:>4} | {max_f1s.mean()*100:7.2f}% | {mean_f1s.mean()*100:7.2f}% "
          f"| {diff*100:+6.2f} | {p_val:13.4f} {flag}")

print("\n* = p < 0.05 (uncorrected). With 7 comparisons, Bonferroni threshold is p < 0.0071")

   N |   max F1 |  mean F1 |    diff |  p (two-sided)
----------------------------------------------------------
   1 |   91.93% |   91.93% |  +0.00 |           nan 
   2 |   92.39% |   91.71% |  +0.68 |        0.3739 
   4 |   92.80% |   91.07% |  +1.73 |        0.1808 
   8 |   90.98% |   91.72% |  -0.74 |        0.3739 
  16 |   91.08% |   91.07% |  +0.01 |        0.9976 
  32 |   88.88% |   89.48% |  -0.60 |        0.6996 
  64 |   88.05% |   90.30% |  -2.25 |        0.3227 

* = p < 0.05 (uncorrected). With 7 comparisons, Bonferroni threshold is p < 0.0071
